# Scaled GPT-2 Training Pipeline on a Multi-GPU Infrastructure

In this colab is an end-to-end distributed pre-training and state management custom GPT-2 architecture implementation, optimized for scaled data streaming and checkpointing for hardware and system failures. Trained on the wiki40b dataset.

## 1. Multi-GPU Orchestration & Gradient Accumulation Strategy

In [ ]:
import tensorflow as tf

# 1. Initialize multi-GPU tensorflow object to handle distributing training across GPUs
strategy = tf.distribute.MirroredStrategy()
print(f'Number of GPUs in sync: {strategy.num_replicas_in_sync}')

# Scale the batch size by the number of GPUs
GLOBAL_BATCH_SIZE = 8 * strategy.num_replicas_in_sync

# Gradient Accumulation Parameters
ACCUMULATION_STEPS = 8 # Accumulate 8 steps to simulate a 64 batch size per GPU
EFFECTIVE_BATCH_SIZE = GLOBAL_BATCH_SIZE * ACCUMULATION_STEPS

print(f'Global Batch Size (Per Step): {GLOBAL_BATCH_SIZE}')
print(f'Effective Batch Size with Accumulation: {EFFECTIVE_BATCH_SIZE}')


Number of GPUs in sync: 8
Global Batch Size (Per Step): 64
Effective Batch Size with Accumulation: 512


## 2. Hardware Diagnostics

In [ ]:
import tensorflow as tf

# Check available physical devices
gpus = tf.config.list_physical_devices('GPU')
print(f"Physical GPUs detected: {len(gpus)}")
for gpu in gpus:
    print(f" - {gpu}")

# Confirm strategy details
print(f"\nStrategy num_replicas_in_sync: {strategy.num_replicas_in_sync}")

Physical GPUs detected: 8
 - PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')
 - PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')
 - PhysicalDevice(name='/physical_device:GPU:2', device_type='GPU')
 - PhysicalDevice(name='/physical_device:GPU:3', device_type='GPU')
 - PhysicalDevice(name='/physical_device:GPU:4', device_type='GPU')
 - PhysicalDevice(name='/physical_device:GPU:5', device_type='GPU')
 - PhysicalDevice(name='/physical_device:GPU:6', device_type='GPU')
 - PhysicalDevice(name='/physical_device:GPU:7', device_type='GPU')

Strategy num_replicas_in_sync: 8


## 3. High-Throughput Streaming Data Pipeline & Tokenization

In [ ]:
# Load dataset

import tensorflow as tf
import tensorflow_datasets as tfds
import keras_hub

# Load Wiki40b (English) natively from TFDS internal mirrors
ds_data = tfds.load('wiki40b/en', split='train')

GPT2_T = 256 # Sequence length

# Instantiate official GPT-2 BPE Tokenizer from Keras Hub
print("Loading Keras Hub GPT-2 Tokenizer...")
tokenizer = keras_hub.models.Tokenizer.from_preset("gpt2_base_en")
gpt2_vocab_size = tokenizer.vocabulary_size()
print(f"GPT-2 Vocab Size: {gpt2_vocab_size}")

def prepare_dataset(example):
    # Tokenize the text natively
    tokens = tokenizer(example['text'])
    tokens = tf.cast(tokens, tf.int32)

    # Truncate or pad to exactly GPT2_T + 1 length
    seq_length = GPT2_T + 1
    tokens = tokens[:seq_length]
    pad_length = tf.maximum(0, seq_length - tf.shape(tokens)[0])

    # Pad with 0s if the sequence is shorter than GPT2_T + 1
    tokens = tf.pad(tokens, [[0, pad_length]], constant_values=0)

    inputs = tokens[:-1]
    targets = tokens[1:]

    # Ensure static shapes for Auto-XLA optimization
    inputs.set_shape([GPT2_T])
    targets.set_shape([GPT2_T])

    return inputs, targets

# Create token streaming pipeline
# OPTIMIZATION: deterministic=False allows parallel workers to fetch and tokenize out-of-order, massively speeding up the pipeline
train_ds_streaming = (
    ds_data
    .map(prepare_dataset, num_parallel_calls=tf.data.AUTOTUNE, deterministic=False)
    .batch(GLOBAL_BATCH_SIZE, drop_remainder=True)
    .prefetch(tf.data.AUTOTUNE)
)

print("Streaming dataset ready! (Replaces in-memory arrays)")


Loading Keras Hub GPT-2 Tokenizer...
GPT-2 Vocab Size: 50257
Streaming dataset ready! (Replaces in-memory arrays)


## 4. Dataset Metadata & Asset Validation

In [ ]:
import tensorflow_datasets as tfds

# Get builder information for wiki40b/en without fully downloading it
builder = tfds.builder('wiki40b/en')

# Display dataset metadata
print(f"Total Dataset Size: {builder.info.dataset_size}")
if 'train' in builder.info.splits:
    print(f"Number of training examples: {builder.info.splits['train'].num_examples}")


Total Dataset Size: 9.91 GiB
Number of training examples: 2926536


## 5. Numerical Stability: Mixed Precision Configuration

In [ ]:
# Mixed precision

from tensorflow.keras import mixed_precision

# Upgraded to mixed_bfloat16 to prevent 'inf' loss (numerical overflow)
policy = mixed_precision.Policy('mixed_bfloat16')
mixed_precision.set_global_policy(policy)

print('Compute dtype: %s' % policy.compute_dtype)
print('Variable dtype: %s' % policy.variable_dtype)

Compute dtype: bfloat16
Variable dtype: float32


## 6. Model Architecture

In [ ]:
# Model build

# GPT-2 Small Hyperparameters
gpt2_d_model = 768
gpt2_num_layers = 12
gpt2_num_heads = 12
gpt2_ffn_dim = 3072 # Typically 4x d_model

# Everything must be inside the scope to be mirrored across GPUs
with strategy.scope():
    gpt2_input = tf.keras.Input(shape=(GPT2_T,))

    # 1. Embeddings (Separated to access weights for Weight Tying)
    word_embedding_layer = tf.keras.layers.Embedding(gpt2_vocab_size, gpt2_d_model, mask_zero=True)
    position_embedding_layer = tf.keras.layers.Embedding(GPT2_T, gpt2_d_model)

    # Broadcast position embeddings across batch
    length = tf.shape(gpt2_input)[-1]
    positions = tf.range(start=0, limit=length, delta=1)
    x = word_embedding_layer(gpt2_input) + position_embedding_layer(positions)
    x = tf.keras.layers.Dropout(0.1)(x)

    # NanoGPT style Pre-Norm & GELU blocks
    def nanogpt_attention_block(x):
        nx = tf.keras.layers.LayerNormalization(epsilon=1e-5)(x) # Pre-Norm
        attn = tf.keras.layers.MultiHeadAttention(
            num_heads=gpt2_num_heads, key_dim=gpt2_d_model // gpt2_num_heads
        )(nx, nx, use_causal_mask=True)
        return tf.keras.layers.Add()([x, tf.keras.layers.Dropout(0.1)(attn)])

    def nanogpt_ffn_block(x):
        nx = tf.keras.layers.LayerNormalization(epsilon=1e-5)(x) # Pre-Norm
        ffn = tf.keras.layers.Dense(gpt2_ffn_dim, activation='gelu')(nx) # GELU Activation
        ffn = tf.keras.layers.Dense(gpt2_d_model)(ffn)
        return tf.keras.layers.Add()([x, tf.keras.layers.Dropout(0.1)(ffn)])

    # 2. 12 Transformer Blocks
    for _ in range(gpt2_num_layers):
        x = nanogpt_attention_block(x)
        x = nanogpt_ffn_block(x)

    x = tf.keras.layers.LayerNormalization(epsilon=1e-5)(x) # Final norm before output

    # 3. Output Projection (Weight Tying)
    class TiedOutputProjection(tf.keras.layers.Layer):
        def __init__(self, embedding_layer, **kwargs):
            super().__init__(**kwargs)
            self.embedding_layer = embedding_layer

        def call(self, inputs):
            # Multiply by the transposed embedding weights to get logits
            # Cast the float32 weights to the compute dtype (bfloat16) to avoid type mismatch
            return tf.matmul(inputs, tf.cast(self.embedding_layer.embeddings, inputs.dtype), transpose_b=True)

    logits = TiedOutputProjection(word_embedding_layer)(x)
    gpt2_outputs = tf.keras.layers.Activation('softmax')(logits)

    # 4. Optimizer and Compilation
    ## Set learning rate with an initial warm up
    gpt2_lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=2.5e-4,
        warmup_target=2.5e-4,
        warmup_steps=1000,
        decay_steps=5000,
        alpha=0.1
    )

    gpt2_optimizer = tf.keras.optimizers.AdamW(
        learning_rate=gpt2_lr_schedule,
        weight_decay=0.01,
        global_clipnorm=1.0, # Prevent exploding gradients
    )

    gpt2_model = tf.keras.Model(inputs=gpt2_input, outputs=gpt2_outputs)
    gpt2_model.compile(optimizer=gpt2_optimizer, loss='sparse_categorical_crossentropy', jit_compile=False)


## 7. Distributed Mathematical Engine: Masked Loss & Gradient Math

In [ ]:
# Distributed Training Setup
import tensorflow as tf

with strategy.scope():
    # 1. Define loss for distributed training
    loss_object = tf.keras.losses.SparseCategoricalCrossentropy(
        from_logits=False, reduction=tf.keras.losses.Reduction.NONE)

    def compute_loss(labels, predictions):
        per_example_loss = loss_object(labels, predictions)

        # Mask out padding tokens (token ID 0)
        mask = tf.cast(labels != 0, per_example_loss.dtype)
        per_example_loss = per_example_loss * mask

        # 1. Sum the loss locally for this specific batch
        local_loss_sum = tf.reduce_sum(per_example_loss)

        # 2. Count exactly how many real tokens are in this batch
        local_active_tokens = tf.reduce_sum(mask)

        # 3. Compute the true average loss (add 1e-5 to prevent division by zero on empty batches)
        local_average_loss = local_loss_sum / (local_active_tokens + 1e-5)

        # 4. Scale by num_replicas_in_sync because your outer loop uses tf.distribute.ReduceOp.SUM
        # If we don't divide by the number of GPUs here, the outer loop will add them together
        # and artificially inflate your loss!
        return local_average_loss / strategy.num_replicas_in_sync

    # 2. Create Gradient Accumulators
    gradient_accumulators = [tf.Variable(tf.zeros_like(v), trainable=False) for v in gpt2_model.trainable_variables]

    def reset_accumulators():
        for acc in gradient_accumulators:
            acc.assign(tf.zeros_like(acc))

    # 3. Math Step: Compute and Accumulate Gradients locally
    @tf.function
    def forward_backward_pass(inputs, targets):
        with tf.GradientTape() as tape:
            predictions = gpt2_model(inputs, training=True)
            loss = compute_loss(targets, predictions)
            # Scale loss down by accumulation steps so the gradients sum up correctly
            scaled_loss = loss / tf.cast(ACCUMULATION_STEPS, loss.dtype)

        gradients = tape.gradient(scaled_loss, gpt2_model.trainable_variables)

        # Accumulate gradients inside the accumulator variables
        for acc, grad in zip(gradient_accumulators, gradients):
            if grad is not None:
                acc.assign_add(grad)

        return loss

    # 4. Apply Step
    def apply_accumulated_gradients():
        gpt2_optimizer.apply_gradients(zip(gradient_accumulators, gpt2_model.trainable_variables))

    @tf.function
    def distributed_accumulate_step(dist_inputs, dist_targets):
        per_replica_losses = strategy.run(forward_backward_pass, args=(dist_inputs, dist_targets))
        return strategy.reduce(tf.distribute.ReduceOp.SUM, per_replica_losses, axis=None)

    @tf.function
    def distributed_apply_and_reset():
        strategy.run(apply_accumulated_gradients)
        strategy.run(reset_accumulators)



## 8. State Management: Checkpoint Configuration

In [ ]:
# Checkpoint save
import os

# Directory to save checkpoints
checkpoint_dir = [add checkpoint directory file path]

# Trackers that will be saved in the checkpoint
global_step = tf.Variable(0, dtype=tf.int64, trainable=False)
current_epoch = tf.Variable(0, dtype=tf.int64, trainable=False)

checkpoint = tf.train.Checkpoint(
    optimizer=gpt2_optimizer,
    model=gpt2_model,
    global_step=global_step,
    current_epoch=current_epoch
)

manager = tf.train.CheckpointManager(checkpoint, directory=checkpoint_dir, max_to_keep=3)

## 9. Environment Sanitization: Stale Checkpoint Purge

In [ ]:
# # Delete checkpoints
# import tensorflow as tf

# # 1. Delete the existing checkpoint directory
# if tf.io.gfile.exists(checkpoint_dir):
#     print(f"Deleting old checkpoints at {checkpoint_dir}...")
#     tf.io.gfile.rmtree(checkpoint_dir)
#     print("Checkpoints successfully deleted.")
# else:
#     print("No checkpoints found to delete.")

# # 2. Re-create the empty directory for future saves
# tf.io.gfile.makedirs(checkpoint_dir)

# # 3. Reset the tracking variables
# global_step.assign(0)
# current_epoch.assign(0)

# print("Ready to start fresh!")

Deleting old checkpoints at /cns/na-d/home/risapark/gpt2_checkpoints...
Checkpoints successfully deleted.
Ready to start fresh!


## 10. Fault Tolerance: Automated Checkpoint Recovery

In [ ]:
# Checkpoint auto-resume
# Re-initialize the manager to ensure it reflects the current state of the directory
manager = tf.train.CheckpointManager(checkpoint, directory=checkpoint_dir, max_to_keep=3)

if manager.latest_checkpoint:
    checkpoint.restore(manager.latest_checkpoint)
    print(f"Restored from {manager.latest_checkpoint}")
    print(f"Resuming at Epoch {current_epoch.numpy() + 1}, Batch {global_step.numpy()}")
else:
    print("No checkpoint found. Starting fresh from Epoch 1, Batch 0.")

No checkpoint found. Starting fresh from Epoch 1, Batch 0.


## 11. Training Loop & Throughput Tracking

In [ ]:
# Training loop

import time
import tensorflow as tf

save_every_n_batches = 1000
total_epochs = 10
tokens_per_batch = GLOBAL_BATCH_SIZE * GPT2_T

start_epoch = int(current_epoch.numpy())

with strategy.scope():
    for epoch in range(start_epoch, total_epochs):
        print(f"\nEpoch {epoch+1}/{total_epochs}")
        start_time = time.time()
        total_loss = 0.0

        strategy.run(reset_accumulators)

        batches_to_skip = int(global_step.numpy())
        num_batches = batches_to_skip

        if batches_to_skip > 0:
          print(f"Fast-forwarding dataset training by {batches_to_skip} batches...")
          epoch_ds = train_ds_streaming.skip(batches_to_skip)
        else:
          epoch_ds = train_ds_streaming

        dist_epoch_dataset = strategy.experimental_distribute_dataset(epoch_ds)

        for dist_inputs, dist_targets in dist_epoch_dataset:
            step_start_time = time.time()

            loss = distributed_accumulate_step(dist_inputs, dist_targets)
            total_loss += float(loss)

            num_batches += 1
            global_step.assign(num_batches)

            if num_batches % ACCUMULATION_STEPS == 0:
                distributed_apply_and_reset()

            step_time = time.time() - step_start_time
            tokens_per_sec = tokens_per_batch / step_time if step_time > 0 else 0

            print(f" Batch {num_batches} Loss: {float(loss):.4f} - Time: {step_time:.2f}s - Tokens/Sec: {tokens_per_sec:.2f}")

            if num_batches % save_every_n_batches == 0:
                manager.save(checkpoint_number=num_batches)

        if num_batches % ACCUMULATION_STEPS != 0:
            distributed_apply_and_reset()

        current_epoch.assign_add(1)
        global_step.assign(0)
        save_path = manager.save()

        epoch_time = time.time() - start_time
        print(f"Epoch {epoch+1} finished in {epoch_time:.2f}s")
        print(f"Total loss: {total_loss:.4f}")


Epoch 1/10
Fast-forwarding dataset training by 21000 batches...
 Batch 21001 Loss: 6.0312 - Time: 0.29s - Tokens/Sec: 55830.87
 Batch 21002 Loss: 6.0625 - Time: 0.29s - Tokens/Sec: 55886.49
 Batch 21003 Loss: 6.0938 - Time: 0.29s - Tokens/Sec: 57017.59
 Batch 21004 Loss: 6.0938 - Time: 0.29s - Tokens/Sec: 57434.32
 Batch 21005 Loss: 6.1250 - Time: 0.29s - Tokens/Sec: 56999.00
 Batch 21006 Loss: 6.1562 - Time: 0.29s - Tokens/Sec: 56512.91
 Batch 21007 Loss: 6.0312 - Time: 0.28s - Tokens/Sec: 57529.91
 Batch 21008 Loss: 6.1250 - Time: 0.36s - Tokens/Sec: 46014.93
 Batch 21009 Loss: 6.0938 - Time: 0.29s - Tokens/Sec: 57256.97
 Batch 21010 Loss: 6.1250 - Time: 0.29s - Tokens/Sec: 57026.96
 Batch 21011 Loss: 6.1250 - Time: 0.29s - Tokens/Sec: 57383.92
 Batch 21012 Loss: 6.1562 - Time: 0.29s - Tokens/Sec: 57402.42
 Batch 21013 Loss: 6.0938 - Time: 0.29s - Tokens/Sec: 57372.42
 Batch 21014 Loss: 6.1250 - Time: 0.29s - Tokens/Sec: 57030.13
 Batch 21015 Loss: 6.0312 - Time: 0.29s - Tokens/Sec:

## 12. Emergency Artifact Persistence: Direct Weight Export

In [ ]:
# Manual model weights save
import tensorflow as tf
import os
import time

# FIX: Append a unique timestamp to the filename so we NEVER overwrite previous backups
timestamp = int(time.time())
emergency_save_path = f"add checkpoint emergency file path"

# 1. Get the parent directory path
save_dir = os.path.dirname(emergency_save_path)

# 2. Force create the directory (and any missing parent directories)
if not tf.io.gfile.exists(save_dir):
    print(f"Creating missing directory: {save_dir}")
    tf.io.gfile.makedirs(save_dir)

Saving model weights...
Success! Weights safely backed up to /cns/na-d/home/risapark/gpt2_emergency_backup.weights.h5


## 13. Inference

In [ ]:
#Inference

import numpy as np
import tensorflow as tf

def generate_gpt2_text(seed_text, gen_length=50, temperature=0.7):
    print(f"--- Generating from GPT-2 Scale Model (T={GPT2_T}) ---")

    # 1. Vectorize seed text using keras_hub tokenizer
    # The tokenizer returns a list natively when given a string
    input_tokens = tokenizer(seed_text)
    if isinstance(input_tokens, tf.Tensor):
        input_tokens = input_tokens.numpy().tolist()

    for _ in range(gen_length):
        # 2. Pad or truncate to the context window (GPT2_T)
        input_padded = tf.keras.preprocessing.sequence.pad_sequences(
            [input_tokens], maxlen=GPT2_T, padding='pre', truncating='pre'
        )

        # 3. Predict the next token using the trained gpt2_model
        preds = gpt2_model.predict(input_padded, verbose=0)[0, -1, :]

        # 4. Apply temperature sampling for variety
        preds = preds / (np.sum(preds) + 1e-8) # Normalize scores
        logits = np.log(preds + 1e-8) / temperature
        next_token_id = tf.random.categorical([logits], num_samples=1)[0, 0].numpy()

        # Append chosen token ID
        input_tokens.append(next_token_id)

    # 5. Decode all tokens back to text cleanly
    # Convert back to tensor for detokenization if needed by the keras_hub tokenizer
    result_text = tokenizer.detokenize(input_tokens)
    if isinstance(result_text, tf.Tensor):
        result_text = result_text.numpy().decode('utf-8')

    return result_text

# --- TEST THE GPT-2 INFERENCE ---
prompt = "The secret to a happy life is"
print(generate_gpt2_text(prompt, gen_length=40, temperature=0.6))

--- Generating from GPT-2 Scale Model (T=256) ---
The secret to a happy life is ofART,,
 and of
ARTART_ ofART_,, in the_ST
 priced ofTerry

,inar OriThe Award ofST nesting
 and
 ofARTART
